Train two independent Poisson regressions: one for home goals, one for away goals.

Each model uses only pre-kickoff information. Matches from 2000 through 2023 are used for fitting. Everything from 2024 onward is held out for validation later.

In [1]:
from pathlib import Path

import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

matches = pd.read_csv("../data/training/training_matches.csv", parse_dates=["date"])
matches["elo_diff"] = matches["home_elo_pre"] - matches["away_elo_pre"]
matches["neutral"] = matches["neutral"].astype(int)

print(matches.shape)
print(matches["date"].min().date(), "->", matches["date"].max().date())
matches.head()

(30020, 14)
1884-01-26 -> 2026-03-31


,date,home_team,away_team,home_confederation,away_confederation,home_elo_pre,away_elo_pre,city,country,neutral,tournament_weight,home_score,away_score,elo_diff
0,1884-01-26,Northern Ireland,Scotland,Unknown,UEFA,1423.706852,1664.788176,Belfast,Ireland,0,2,0.0,5.0,-241.081323
1,1884-02-09,Wales,Northern Ireland,Unknown,Unknown,1419.711430,1405.260637,Wrexham,Wales,0,2,6.0,0.0,14.450793
2,1884-02-23,Northern Ireland,England,Unknown,UEFA,1383.521753,1491.793542,Belfast,Ireland,0,2,1.0,8.0,-108.271789
3,1884-03-15,Scotland,England,UEFA,UEFA,1683.234391,1524.740170,Glasgow,Scotland,0,2,1.0,0.0,158.494221
4,1884-03-17,Wales,England,Unknown,UEFA,1441.450314,1519.213562,Wrexham,Wales,0,2,0.0,4.0,-77.763248


Predictors: home_elo_pre, away_elo_pre, home_confederation, away_confederation, neutral, elo_diff, tournament_weight.

elo_diff is home_elo_pre - away_elo_pre. Confederations are dummy-encoded.

Train: 2000-01-01 through 2023-12-31. Validation: 2024-01-01 through 2026. The cell below splits them into separate files and fails if any 2024-2026 match is in train.

In [6]:
TRAIN_END = pd.Timestamp("2024-01-01")

train = matches[(matches["date"] >= "2000-01-01") & (matches["date"] < TRAIN_END)].copy()
validation = matches[matches["date"] >= TRAIN_END].copy()

checks = {
    "Train ends before 2024-01-01": train["date"].max() < TRAIN_END,
    "Validation starts on/after 2024-01-01": validation["date"].min() >= TRAIN_END,
    "No shared rows": train.index.intersection(validation.index).empty,
}

print("Checks (notebook stops if any fail)")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
    assert passed, name

train_path = Path("../data/training/train.csv")
validation_path = Path("../data/training/validation.csv")
train.to_csv(train_path, index=False)
validation.to_csv(validation_path, index=False)

print(f"\nTrain:      {len(train):>5}  {train['date'].min().date()} -> {train['date'].max().date()}")
print(f"Validation: {len(validation):>5}  {validation['date'].min().date()} -> {validation['date'].max().date()}")
print("Train years:", sorted(train["date"].dt.year.unique())[:3], "...", sorted(train["date"].dt.year.unique())[-3:])
print("Validation years:", sorted(validation["date"].dt.year.unique()))
print(f"\nSaved {train_path}")
print(f"Saved {validation_path}")
print("\nTrain mean goals")
print(train[["home_score", "away_score"]].mean())

Checks (notebook stops if any fail)
PASS: Train ends before 2024-01-01
PASS: Validation starts on/after 2024-01-01
PASS: No shared rows

Train:      14344  2000-01-22 -> 2023-12-02
Validation:  1839  2024-01-12 -> 2026-03-31
Train years: [np.int32(2000), np.int32(2001), np.int32(2002)] ... [np.int32(2021), np.int32(2022), np.int32(2023)]
Validation years: [np.int32(2024), np.int32(2025), np.int32(2026)]

Saved ../data/training/train.csv
Saved ../data/training/validation.csv

Train mean goals
home_score    1.674847
away_score    1.131902
dtype: float64


Same design matrix for both models. Home goals and away goals are fit as separate Poisson GLMs.

In [7]:
formula_rhs = (
    "home_elo_pre + away_elo_pre + C(home_confederation) + C(away_confederation) "
    "+ neutral + elo_diff + tournament_weight"
)

home_model = smf.glm(
    formula=f"home_score ~ {formula_rhs}",
    data=train,
    family=sm.families.Poisson(),
).fit()

away_model = smf.glm(
    formula=f"away_score ~ {formula_rhs}",
    data=train,
    family=sm.families.Poisson(),
).fit()

print(home_model.summary())
print(away_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_score   No. Observations:                14344
Model:                            GLM   Df Residuals:                    14327
Model Family:                 Poisson   Df Model:                           16
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -22650.
Date:                Sun, 16 Aug 2026   Deviance:                       18337.
Time:                        12:48:31   Pearson chi2:                 1.78e+04
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3846
Covariance Type:            nonrobust                                         
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Interc

This data allows us to use confederation to influence the regression output. A higher |z| value would indicate that if the team is from that specific confederation, it affects the number of goals in the match. According to the data above, this would be CAF, UEFA, OFC, and Unknown. 

OFC (Oceania) looks wild because there are so few teams. 

In-sample check. Expected goals should match the training averages. Validation MAE is in the next section.

In [8]:
train["home_lambda"] = home_model.fittedvalues
train["away_lambda"] = away_model.fittedvalues

print("Actual vs fitted mean goals (train)")
print(train[["home_score", "home_lambda", "away_score", "away_lambda"]].mean())

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)
home_model.save(models_dir / "poisson_home.pickle")
away_model.save(models_dir / "poisson_away.pickle")
print(f"\nSaved {models_dir / 'poisson_home.pickle'}")
print(f"Saved {models_dir / 'poisson_away.pickle'}")

train[
    ["date", "home_team", "away_team", "home_score", "home_lambda", "away_score", "away_lambda"]
].head(10)

Actual vs fitted mean goals (train)
home_score     1.674847
home_lambda    1.674847
away_score     1.131902
away_lambda    1.131902
dtype: float64

Saved ../models/poisson_home.pickle
Saved ../models/poisson_away.pickle


,date,home_team,away_team,home_score,home_lambda,away_score,away_lambda
13837,2000-01-22,Ghana,Cameroon,1.0,1.203850,1.0,0.857394
13838,2000-01-23,Vietnam,Guam,11.0,1.954086,0.0,0.774841
13839,2000-01-23,South Africa,Gabon,3.0,1.519298,1.0,0.882793
13840,2000-01-23,Nigeria,Tunisia,4.0,1.272710,2.0,0.767116
13841,2000-01-23,Egypt,Zambia,2.0,1.260022,0.0,1.044169
13842,2000-01-23,China PR,Philippines,8.0,7.057396,0.0,0.253560
13843,2000-01-24,DR Congo,Algeria,0.0,1.168980,0.0,0.985907
13844,2000-01-24,Ivory Coast,Togo,1.0,1.873512,1.0,0.745287
13845,2000-01-25,Burkina Faso,Senegal,1.0,1.063666,3.0,1.309750
13846,2000-01-25,Morocco,Congo,1.0,2.131949,0.0,0.598350


Validate on 2024-2026 matches the model never trained on. MAE is the average |actual goals − predicted expected goals| for home, away, and both combined.

In [9]:
validation = validation.copy()
validation["home_lambda"] = home_model.predict(validation)
validation["away_lambda"] = away_model.predict(validation)

def mae(actual, predicted):
    return (actual - predicted).abs().mean()

val_home_mae = mae(validation["home_score"], validation["home_lambda"])
val_away_mae = mae(validation["away_score"], validation["away_lambda"])
val_all_mae = mae(
    pd.concat([validation["home_score"], validation["away_score"]], ignore_index=True),
    pd.concat([validation["home_lambda"], validation["away_lambda"]], ignore_index=True),
)
train_home_mae = mae(train["home_score"], train["home_lambda"])
train_away_mae = mae(train["away_score"], train["away_lambda"])
train_all_mae = mae(
    pd.concat([train["home_score"], train["away_score"]], ignore_index=True),
    pd.concat([train["home_lambda"], train["away_lambda"]], ignore_index=True),
)

print(f"Validation matches: {len(validation)}  {validation['date'].min().date()} -> {validation['date'].max().date()}")
print("\nMAE (goals)")
print(f"{'':<12}{'home':>8}{'away':>8}{'all':>8}")
print(f"{'Train':<12}{train_home_mae:>8.3f}{train_away_mae:>8.3f}{train_all_mae:>8.3f}")
print(f"{'Validation':<12}{val_home_mae:>8.3f}{val_away_mae:>8.3f}{val_all_mae:>8.3f}")
print("\nValidation mean goals (actual vs predicted)")
print(validation[["home_score", "home_lambda", "away_score", "away_lambda"]].mean())

validation[
    ["date", "home_team", "away_team", "home_score", "home_lambda", "away_score", "away_lambda"]
].head(10)

Validation matches: 1839  2024-01-12 -> 2026-03-31

MAE (goals)
                home    away     all
Train          1.068   0.865   0.967
Validation     1.047   0.863   0.955

Validation mean goals (actual vs predicted)
home_score     1.595976
home_lambda    1.696692
away_score     1.164763
away_lambda    1.164640
dtype: float64


,date,home_team,away_team,home_score,home_lambda,away_score,away_lambda
28181,2024-01-12,Qatar,Lebanon,3.0,2.525599,0.0,0.518129
28182,2024-01-13,China PR,Tajikistan,0.0,1.521400,0.0,1.106061
28183,2024-01-13,Australia,India,2.0,3.523909,0.0,0.429016
28184,2024-01-13,Uzbekistan,Syria,0.0,2.121557,0.0,0.735798
28185,2024-01-13,Ivory Coast,Guinea-Bissau,2.0,2.165617,0.0,0.470278
28186,2024-01-14,Ghana,Cape Verde,1.0,1.366385,2.0,0.840765
28187,2024-01-14,Egypt,Mozambique,2.0,2.225367,2.0,0.586776
28188,2024-01-14,Nigeria,Equatorial Guinea,1.0,1.718952,1.0,0.929324
28189,2024-01-14,Japan,Vietnam,4.0,3.340719,2.0,0.413795
28190,2024-01-14,Iran,Palestine,4.0,2.725723,1.0,0.533099


On home training data, the model is off by 1.068 goals on average. On home validation data it is off by 1.047 goals on average. 

With away training data, the model is off by 0.865 on average. On away validation data it is off by 0.863 goals on average. 

My model predicted 1.70 goals for the home team on average, with actual home team goals being 1.60. The difference is small. The away goal predict vs actual are almost spot on. This tells us that the model is accurate.